# Homework #5 – Deploying Machine Learning Models

Run this notebook from `cohorts/2026/homework/05-deployment`, after:

```bash
uv sync --locked
uv run python smoke_test.py
```

Start Jupyter with `uv run --with jupyter jupyter lab` so the kernel uses the locked environment (scikit-learn 1.7.2).

## Artifact check
`pipeline.bin` must have the checksum from the homework.

In [1]:
import hashlib, json, pickle, re, subprocess, sys

EXPECTED_SHA = '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1'

def sha256(path):
    with open(path, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()

model_sha = sha256('pipeline.bin')
print('pipeline.bin sha256:', model_sha)
assert model_sha == EXPECTED_SHA, 'pipeline.bin does not match the frozen artifact!'
print('Checksum OK')

pipeline.bin sha256: 1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1
Checksum OK


## Question 1 – environment check (not graded)

In [2]:
print(subprocess.run(['uv', '--version'], capture_output=True, text=True).stdout)

uv 0.8.22 (ade2bdbd2 2025-09-23)



## Question 2 – locked dependency
Read the scikit-learn version from `pyproject.toml` and `uv.lock`.

In [3]:
pyproject = open('pyproject.toml').read()
pinned = re.search(r'scikit-learn==([\d.]+)', pyproject).group(1)

lock = open('uv.lock').read()
locked = re.search(r'name = "scikit-learn"\nversion = "([\d.]+)"', lock).group(1)

import sklearn
print('pyproject.toml:', pinned)
print('uv.lock:       ', locked)
print('kernel uses:   ', sklearn.__version__)
print('Answer Q2:', locked)

pyproject.toml: 1.7.2
uv.lock:        1.7.2
kernel uses:    1.7.2
Answer Q2: 1.7.2


## Question 3 – load the model

In [4]:
lead_q3 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

with open('pipeline.bin', 'rb') as f:
    pipeline = pickle.load(f)

proba_q3 = pipeline.predict_proba([lead_q3])[0, 1]
print('Probability of conversion:', proba_q3)
print('Answer Q3:', round(proba_q3, 3))

Probability of conversion: 0.5329944186432507
Answer Q3: 0.533


## Question 4 – serve the model

In a separate terminal (same folder):

```bash
uv run uvicorn predict:app --host 0.0.0.0 --port 9696
```

In [5]:
import requests

URL = 'http://localhost:9696'

lead_q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

print('Health:', requests.get(f'{URL}/health').json())

response = requests.post(f'{URL}/predict', json=lead_q4).json()
print('Response:', response)

p = response['conversion_probability']
assert 0 <= p <= 1
assert isinstance(response['conversion'], bool)
print('Answer Q4:', round(p, 3))

Health: {'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}
Response: {'conversion_probability': 0.769799, 'conversion': True}
Answer Q4: 0.77


## Question 5 – container configuration

In [6]:
dockerfile = open('Dockerfile').read()
base_image = re.search(r'^FROM\s+([^\s@]+)', dockerfile, re.M).group(1)
print('Answer Q5:', base_image)

Answer Q5: python:3.11.15-slim-bookworm


## Question 6 – run the container

Stop the uvicorn server from Q4 first (both use port 9696), then in a terminal:

```bash
docker build -t zoomcamp-model:2026-hw5 .
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
```

In [7]:
print(subprocess.run([sys.executable, 'q6_test.py'], capture_output=True, text=True).stdout)

{'conversion_probability': 0.769799, 'conversion': True}



In [8]:
health = requests.get(f'{URL}/health').json()
print('Health:', health)
assert health['status'] == 'ok'

metadata = json.load(open('model_metadata.json'))
print('Metadata model_sha256 matches artifact:', metadata['model_sha256'] == EXPECTED_SHA)

response = requests.post(f'{URL}/predict', json=lead_q4).json()
print('Response:', response)
print('Answer Q6:', round(response['conversion_probability'], 3))

Health: {'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}
Metadata model_sha256 matches artifact: True
Response: {'conversion_probability': 0.769799, 'conversion': True}
Answer Q6: 0.77
